In [1]:
print("all ok")

all ok


Install Dependencies

In [ ]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install datasets pymupdf huggingface_hub

HF Login

In [3]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("✅ Hugging Face Authenticated")

✅ Hugging Face Authenticated


Imports & Config

In [4]:
import os
import fitz
import torch

from datasets import load_dataset

from unsloth import FastLanguageModel

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

from huggingface_hub import (
    create_repo,
    upload_folder,
)

MAX_SEQ_LENGTH = 2048

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"

STAGE1_REPO = (
    "Pzazz55/insurance-ai-assistant-finetuning-stage1"
)

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/outputs_raw", exist_ok=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Create HF Repo

In [11]:
from huggingface_hub import HfApi

api = HfApi()

repos = [
    "Pzazz55/insurance-ai-assistant-finetuning-stage1",
    "Pzazz55/insurance-ai-assistant-finetuning-stage2",
    "Pzazz55/insurance-ai-assistant-finetuning-stage3",
]

for repo in repos:
    api.delete_repo(
        repo_id=repo,
        repo_type="model",
        missing_ok=True
    )
    print(f"✅ Deleted (or did not exist): {repo}")

✅ Deleted (or did not exist): Pzazz55/insurance-ai-assistant-finetuning-stage1
✅ Deleted (or did not exist): Pzazz55/insurance-ai-assistant-finetuning-stage2
✅ Deleted (or did not exist): Pzazz55/insurance-ai-assistant-finetuning-stage3


In [12]:
create_repo(
    repo_id=STAGE1_REPO,
    exist_ok=True
)

print("✅ Stage 1 Repo Ready")

✅ Stage 1 Repo Ready


In [13]:
import os
import shutil
from huggingface_hub import snapshot_download

# Hugging Face Dataset
HF_DATASET_REPO = "Pzazz55/insurance-ai-assistant-finetuning-dataset"

# Destination folder in Colab
INPUT_DATA_DIR = "/content/input_data"

# Clean existing input_data folder
if os.path.exists(INPUT_DATA_DIR):
    shutil.rmtree(INPUT_DATA_DIR)

os.makedirs(INPUT_DATA_DIR, exist_ok=True)

# Download dataset from Hugging Face
dataset_dir = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset"
)

# Copy files into input_data
for item in os.listdir(dataset_dir):
    src = os.path.join(dataset_dir, item)
    dst = os.path.join(INPUT_DATA_DIR, item)

    if item.startswith("."):
        continue

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("✅ Files copied successfully to /content/input_data")

# Verify files
for root, dirs, files in os.walk(INPUT_DATA_DIR):
    for file in files:
        print(os.path.join(root, file))

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

insurance_instruction_dataset.jsonl: 0.00B [00:00, ?B/s]

insurance_rawdata.pdf:   0%|          | 0.00/143k [00:00<?, ?B/s]

insurance_preference_dataset.jsonl: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

✅ Files copied successfully to /content/input_data
/content/input_data/insurance_preference_dataset.jsonl
/content/input_data/insurance_instruction_dataset.jsonl
/content/input_data/insurance_rawdata.pdf


In [7]:
# import os
# import shutil

# repo_url = "https://github.com/Pzazz55/ScriptPython.git"
# repo_dir = "/content/ScriptPython"

# # Corrected path: 'ft' is directly under /content/ScriptPython/
# src_folder = "/content/ScriptPython/ft"
# dst_folder = "/content/input_data"

# # -----------------------------
# # 1. Clean up and Clone repo
# # -----------------------------
# # Remove existing repo_dir to prevent 'destination path already exists' error
# if os.path.exists(repo_dir):
#     shutil.rmtree(repo_dir)

# !git clone {repo_url}

# # -----------------------------
# # 2. Copy ft folder to input_data
# # -----------------------------
# os.makedirs(dst_folder, exist_ok=True)

# if not os.path.exists(src_folder):
#     # Re-raise with a more informative message if the corrected path still doesn't exist
#     raise FileNotFoundError(f"Source folder not found after correction: {src_folder}. Please verify repository structure or intended source.")

# for item in os.listdir(src_folder):
#     s = os.path.join(src_folder, item)
#     d = os.path.join(dst_folder, item)

#     if os.path.isdir(s):
#         shutil.copytree(s, d, dirs_exist_ok=True)
#     else:
#         shutil.copy2(s, d)

# print("Copy completed!")

# # -----------------------------
# # 3. Delete cloned repository
# # -----------------------------
# shutil.rmtree(repo_dir)

# print("Repository deleted, only input_data retained!")

# # -----------------------------
# # 4. Verify
# # -----------------------------
# for root, dirs, files in os.walk(dst_folder):
#     for f in files:
#         print(os.path.join(root, f))

Cloning into 'ScriptPython'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 38 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 128.95 KiB | 7.58 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Copy completed!
Repository deleted, only input_data retained!
/content/input_data/insurance_preference_dataset.jsonl
/content/input_data/insurance_instruction_dataset.jsonl
/content/input_data/insurance_rawdata.pdf


Extract PDF Text

In [14]:
def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = []
    for page in doc:
        text.append(page.get_text())

    return "\n".join(text)

pdf_path = (
    "/content/input_data/insurance_rawdata.pdf"
)

raw_text = extract_pdf_text(pdf_path)

output_txt = (
    "/content/outputs_raw/insurance_rawdata.txt"
)

with open(
    output_txt, "w", encoding="utf-8"
) as f:
    f.write(raw_text)

print("✅ PDF Extracted")

✅ PDF Extracted


Load Dataset

In [9]:
raw_dataset = load_dataset(
    "text",
    data_files={
        "train": output_txt
    }
)

raw_dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2556
    })
})

Load Base Model

In [15]:
model, tokenizer = (
    FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
)

print("✅ Base Model Loaded")

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Base Model Loaded


Attach LoRA

In [16]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("✅ LoRA Attached")

Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA Attached


Tokenize

In [17]:
def tokenize_fn(examples):
    tokenized = tokenizer(
        [text + tokenizer.eos_token for text in examples["text"]],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

tokenized_dataset = raw_dataset["train"].map(
    tokenize_fn,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)

Map:   0%|          | 0/2556 [00:00<?, ? examples/s]

Train Stage 1

In [18]:
tokenized_dataset = tokenized_dataset.with_format("torch")

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        output_dir="outputs_raw",
        save_strategy="no",
        report_to="none",
    ),
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,556 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
5,8.790300
10,6.028400
15,5.483900
20,5.327100
25,4.659700
30,3.969000
35,4.487300
40,3.924000
45,3.701900
50,3.385800


TrainOutput(global_step=60, training_loss=4.802859878540039, metrics={'train_runtime': 121.2814, 'train_samples_per_second': 1.979, 'train_steps_per_second': 0.495, 'total_flos': 19948026304512.0, 'train_loss': 4.802859878540039, 'epoch': 0.09389671361502347})

Save Model

In [19]:
stage1_path = (
    "/content/models/stage1_non_instruction"
)

model.save_pretrained_merged(
    stage1_path,
    tokenizer,
    save_method="merged_16bit"
)

print("✅ Stage 1 Saved")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:32<00:00, 92.59s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:55<00:00, 55.92s/it]


Unsloth: Merge process complete. Saved to `/content/models/stage1_non_instruction`
✅ Stage 1 Saved


Upload To HF

In [20]:
upload_folder(
    repo_id=STAGE1_REPO,
    folder_path=stage1_path,
)

print("✅ Uploaded To Hugging Face")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nstruction/tokenizer.json:   0%|          | 27.9kB / 11.4MB            

  ...ruction/model.safetensors:   2%|1         | 48.0MB / 3.09GB            

✅ Uploaded To Hugging Face


Reload From HF

In [ ]:
del model
del trainer

import gc
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = (
    FastLanguageModel.from_pretrained(
        model_name=STAGE1_REPO,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
)

FastLanguageModel.for_inference(model)

print("✅ Stage 1 Reloaded From HF")

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Evaluation Questions

In [ ]:
evaluation_questions = [
    "What mandatory items must be present in a Middle Market risk submission before it can be processed for binding?",
    "How does a prior history of lapses in commercial auto coverage impact premium calculations and overall risk acceptability?",
    "What specific criteria determine if a commercial property qualifies for highly protected risk (HPR) status?",
    "Explain how aggregate limit extensions operate within a commercial general liability policy during a catastrophic loss year.",
    "What underwriting indicators necessitate the inclusion of a specialized environmental pollution liability rider?",
    "How should an underwriter evaluate the risk profile of a manufacturing plant utilizing aging machinery without telematics tracking?",
    "What are the core differences between a claims-made policy form and an occurrence policy form regarding professional liability?",
    "Under what specific conditions is a retroactive date adjustment permitted on an executive directors and officers (D&O) liability policy?",
    "How does a business interruption policy handle contingent business income losses if a key downstream supplier suffers a fire?",
    "What risk mitigation factors can offset a high experience modification rate (E-Mod) when underwriting worker's compensation?"
]

Alpaca Inference

In [ ]:
def generate_answer(question):

    prompt = f"""
### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.1,
        )

    text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    return text.split(
        "### Response:"
    )[-1].strip()

Generate Report

In [ ]:
# # Generate Report
# markdown = [
#     "# Stage 1 Evaluation Report",
#     "| Question | Model Answer |",
#     "|---|---|"
# ]

# print("🚀 Starting Evaluation Loop...")
# print("-" * 50)

# for idx, question in enumerate(evaluation_questions, 1):
#     # 1. Print the current progress and question to console
#     print(f"\n[Question {idx}/{len(evaluation_questions)}]: {question}")

#     # 2. Generate the model's prediction
#     answer = generate_answer(question)

#     # 3. Print the answer directly to the console
#     print(f"[Model Answer]: {answer}")
#     print("-" * 50)

#     # 4. Append to the markdown file logic (preserving the HTML line breaks for table formatting)
#     markdown.append(
#         f"| {question} | {answer.replace(chr(10), '<br>')} |"
#     )

# # Save the final report
# with open("/content/reports/stage1_evaluation_report.md", "w") as f:
#     f.write("\n".join(markdown))

# print("\n✅ Evaluation complete! Report saved to /content/reports/stage1_evaluation_report.md")

In [ ]:
import os
from huggingface_hub import HfApi

# 1. Define your Hugging Face configuration
HF_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage1"
# HF_TOKEN = "YOUR_HF_WRITE_TOKEN"

# Generate Report Structure
markdown = [
    "# Stage 1 Evaluation Report",
    "",
    "| Question | Base Model Answer |",
    "|---|---|",
]

print("🚀 Starting Evaluation Loop...")
print("-" * 50)

# Loop cleanly through all 10 custom questions
for idx, question in enumerate(evaluation_questions, 1):
    # 1. Print the current progress and question to console
    print(f"\n[Question {idx}/{len(evaluation_questions)}]: {question}")
    print("-" * 50)

    # 2. Generate the model's prediction
    answer = generate_answer(question)

    # 3. Print the answer directly to the console
    print(f"[Model Answer]:\n{answer}")
    print("-" * 50)

    # 4. Append to the markdown file logic (preserving the HTML line breaks for table formatting)
    markdown.append(
        f"| {question} | {answer.replace(chr(10), '<br>')} |"
    )

# Save the final report locally first
local_report_path = "/content/reports/stage1_evaluation_report.md"
os.makedirs(os.path.dirname(local_report_path), exist_ok=True)

with open(local_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(markdown))

print(f"\n✅ Evaluation complete! Local report saved to {local_report_path}")
print("-" * 50)

# 5. Push the file directly to your Hugging Face repository
print(f"📤 Uploading report to Hugging Face Hub ({HF_REPO_ID})...")
try:
    api = HfApi(token=HF_TOKEN)

    # This places the file into a folder called 'results' inside your repo
    path_in_repo = "results/stage1_evaluation_report.md"

    api.upload_file(
        path_or_fileobj=local_report_path,
        path_in_repo=path_in_repo,
        repo_id=HF_REPO_ID,
        repo_type="model" # Change to "dataset" if your repository is a Dataset type instead of a Model type
    )
    print(f"🎉 Success! File saved in HF repo at: results/stage1_evaluation_report.md")

except Exception as e:
    print(f"❌ Hugging Face upload failed. Error details: {e}")
    print("Please double-check your HF Token permissions (must be 'Write') and repo ID structure.")

In [ ]:
import os
from datetime import datetime
from huggingface_hub import HfApi

# ============================================================
# Hugging Face Configuration
# ============================================================
HF_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage1"
# HF_TOKEN = HF_TOKEN

api = HfApi(token=HF_TOKEN)

# ============================================================
# Generate Evaluation Report
# ============================================================
markdown = [
    "# Stage 1 Evaluation Report",
    "",
    f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "| Question | Base Model Answer |",
    "|---|---|",
]

print("🚀 Starting Evaluation Loop...")
print("-" * 80)

for idx, question in enumerate(evaluation_questions, start=1):
    print(f"\n[{idx}/{len(evaluation_questions)}] {question}")
    print("-" * 80)

    answer = generate_answer(question)

    print(answer)
    print("-" * 80)

    markdown.append(
        f"| {question} | {answer.replace(chr(10), '<br>')} |"
    )

# ============================================================
# Save Report Locally
# ============================================================
local_dir = "/content/reports"
os.makedirs(local_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

local_report_path = os.path.join(
    local_dir,
    f"stage1_evaluation_report_{timestamp}.md"
)

with open(local_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(markdown))

print(f"\n✅ Local report saved to:\n{local_report_path}")

# ============================================================
# Create HF Repository (if it doesn't exist)
# ============================================================
try:
    api.create_repo(
        repo_id=HF_REPO_ID,
        repo_type="model",
        exist_ok=True,
        private=False,
    )
except Exception:
    pass

# ============================================================
# Upload Report
# ============================================================
hf_path = f"results/stage1_evaluation_report_{timestamp}.md"

print(f"\n📤 Uploading report to {HF_REPO_ID}...")

try:
    api.upload_file(
        path_or_fileobj=local_report_path,
        path_in_repo=hf_path,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message=f"Add Stage 1 evaluation report ({timestamp})",
    )

    print("\n🎉 Upload Successful!")
    print(f"HF Path : {hf_path}")
    print(f"https://huggingface.co/{HF_REPO_ID}/tree/main/results")

except Exception as e:
    print("\n❌ Upload Failed")
    print(e)